# Track A: Head-to-Head Benchmarking (Dual Cohort Analysis)
### Models: PRIDICT2.0 vs. DeepPrime-base (GenET) vs. DeepPrime6

This notebook performs the head-to-head comparison of PRIDICT2.0, DeepPrime-base (GenET), and DeepPrime6 on `data/rq3_test_library.parquet` containing 2,439 held-out test pegRNAs.

We evaluate performance across two cohorts:
1. **All Edit Types** (N=2,439)
2. **Substitutions Only** (N=861)

The sub-analysis allows us to decouple potential sequence length/formatting alignments from the core **cross-domain generalization gap** caused by scaffold alterations (epegRNAs) and prime editor domain shifts (PE6/PE2max-dRNaseH).

## 1. Setup and Imports

In [20]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr, pearsonr

print("Setup complete.")

Setup complete.


## 2. Load and Merge Predictions

In [21]:
print("Loading datasets...")
gt_df = pd.read_parquet("../data/rq3_test_library.parquet")

dp_base = pd.read_csv("../data/rq3_predictions_dp_base.csv")
dp6_pe6a = pd.read_csv("../data/rq3_predictions_dp6.csv")
dp6_pe6b = pd.read_csv("../data/rq3_predictions_dp6_pe6b.csv")
dp6_pe6c = pd.read_csv("../data/rq3_predictions_dp6_pe6c.csv")
dp6_drn = pd.read_csv("../data/rq3_predictions_dp6_drnaseh.csv")
pridict2 = pd.read_csv("../data/rq3_predictions_pridict2.csv")

dp_base = dp_base[['ID', 'Prediction']].rename(columns={'Prediction': 'pred_dp_base'})
dp6_pe6a = dp6_pe6a[['ID', 'Prediction']].rename(columns={'Prediction': 'pred_dp6_pe6a'})
dp6_pe6b = dp6_pe6b[['ID', 'Prediction']].rename(columns={'Prediction': 'pred_dp6_pe6b'})
dp6_pe6c = dp6_pe6c[['ID', 'Prediction']].rename(columns={'Prediction': 'pred_dp6_pe6c'})
dp6_drn = dp6_drn[['ID', 'Prediction']].rename(columns={'Prediction': 'pred_dp6_drnaseh'})

pridict2 = pridict2[['REF_ID', 'PRIDICT2_0_editing_Score_deep_HEK', 'PRIDICT2_0_editing_Score_deep_K562']].rename(
    columns={
        'PRIDICT2_0_editing_Score_deep_HEK': 'pred_pridict2_hek',
        'PRIDICT2_0_editing_Score_deep_K562': 'pred_pridict2_k562'
    }
)

gt_df = gt_df.reset_index(drop=True)
df = gt_df.copy()

df = df.merge(dp_base, left_on='REF_ID', right_on='ID', how='left').drop(columns=['ID'])
df = df.merge(dp6_pe6a, left_on='REF_ID', right_on='ID', how='left').drop(columns=['ID'])
df = df.merge(dp6_pe6b, left_on='REF_ID', right_on='ID', how='left').drop(columns=['ID'])
df = df.merge(dp6_pe6c, left_on='REF_ID', right_on='ID', how='left').drop(columns=['ID'])
df = df.merge(dp6_drn, left_on='REF_ID', right_on='ID', how='left').drop(columns=['ID'])
df = df.merge(pridict2, on='REF_ID', how='left')

print(f"Merged dataset shape: {df.shape}")

Loading datasets...
Merged dataset shape: (2439, 184)


## 3. Correlation Calculator Function

In [22]:
def calculate_correlations(df, cohort_name):
    domains = {
        'PE2max': ('Normalized+3rep_HEK-M-1-7D+pe_ratio_%', 'pred_dp_base'),
        'PE2max-dRNaseH': ('Normalized+3rep_HEK-M-2-7D+pe_ratio_%', 'pred_dp6_drnaseh'),
        'PE6a': ('Normalized+3rep_HEK-M-3-7D+pe_ratio_%', 'pred_dp6_pe6a'),
        'PE6b': ('Normalized+3rep_HEK-M-4-7D+pe_ratio_%', 'pred_dp6_pe6b'),
        'PE6c': ('Normalized+3rep_HEK-M-5-7D+pe_ratio_%', 'pred_dp6_pe6c')
    }
    
    results = []
    for domain_name, (gt_col, dp6_col) in domains.items():
        cols = list(set([gt_col, 'pred_dp_base', dp6_col, 'pred_pridict2_hek']))
        sub_df = df[cols].dropna()
        n_samples = len(sub_df)
        
        sp_base = spearmanr(sub_df[gt_col], sub_df['pred_dp_base'])[0]
        sp_dp6 = spearmanr(sub_df[gt_col], sub_df[dp6_col])[0] if domain_name != 'PE2max' else np.nan
        sp_pridict = spearmanr(sub_df[gt_col], sub_df['pred_pridict2_hek'])[0]
        
        pe_base = pearsonr(sub_df[gt_col], sub_df['pred_dp_base'])[0]
        pe_dp6 = pearsonr(sub_df[gt_col], sub_df[dp6_col])[0] if domain_name != 'PE2max' else np.nan
        pe_pridict = pearsonr(sub_df[gt_col], sub_df['pred_pridict2_hek'])[0]
        
        results.append({
            'Domain': domain_name,
            'N': n_samples,
            'Spearman_PRIDICT2': sp_pridict,
            'Spearman_DeepPrime_Base': sp_base,
            'Spearman_DeepPrime6': sp_dp6,
            'Pearson_PRIDICT2': pe_pridict,
            'Pearson_DeepPrime_Base': pe_base,
            'Pearson_DeepPrime6': pe_dp6
        })
        
    res_df = pd.DataFrame(results)
    return res_df

## 4. Cohort 1: All Edit Types (N=2,439)

In [23]:
res_all = calculate_correlations(df, "All Edit Types")
res_all

## 5. Cohort 2: Substitutions Only (N=861)

In [24]:
sub_df = df[df['Edit_type'] == 'Sub'].copy()
res_sub = calculate_correlations(sub_df, "Substitutions Only")
res_sub

## 6. Plotting Heatmaps (Spearman & Pearson) & Scatters with white-to-blue theme

In [ ]:
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(18, 11))

# All - Spearman
heatmap_sp_all = res_all.set_index('Domain')[['Spearman_PRIDICT2', 'Spearman_DeepPrime_Base', 'Spearman_DeepPrime6']]
heatmap_sp_all.columns = ['PRIDICT2.0', 'DeepPrime-Base', 'DeepPrime6']
sns.heatmap(heatmap_sp_all, annot=True, cmap="Blues", fmt=".3f", ax=axes[0, 0], cbar_kws={'label': 'Spearman $\\rho$'})
axes[0, 0].set_title("All Edit Types (Spearman $\\rho$)")

# All - Pearson
heatmap_pe_all = res_all.set_index('Domain')[['Pearson_PRIDICT2', 'Pearson_DeepPrime_Base', 'Pearson_DeepPrime6']]
heatmap_pe_all.columns = ['PRIDICT2.0', 'DeepPrime-Base', 'DeepPrime6']
sns.heatmap(heatmap_pe_all, annot=True, cmap="Blues", fmt=".3f", ax=axes[0, 1], cbar_kws={'label': 'Pearson $r$'})
axes[0, 1].set_title("All Edit Types (Pearson $r$)")

# Sub - Spearman
heatmap_sp_sub = res_sub.set_index('Domain')[['Spearman_PRIDICT2', 'Spearman_DeepPrime_Base', 'Spearman_DeepPrime6']]
heatmap_sp_sub.columns = ['PRIDICT2.0', 'DeepPrime-Base', 'DeepPrime6']
sns.heatmap(heatmap_sp_sub, annot=True, cmap="Blues", fmt=".3f", ax=axes[1, 0], cbar_kws={'label': 'Spearman $\\rho$'})
axes[1, 0].set_title("Substitutions Only (Spearman $\\rho$)")

# Sub - Pearson
heatmap_pe_sub = res_sub.set_index('Domain')[['Pearson_PRIDICT2', 'Pearson_DeepPrime_Base', 'Pearson_DeepPrime6']]
heatmap_pe_sub.columns = ['PRIDICT2.0', 'DeepPrime-Base', 'DeepPrime6']
sns.heatmap(heatmap_pe_sub, annot=True, cmap="Blues", fmt=".3f", ax=axes[1, 1], cbar_kws={'label': 'Pearson $r$'})
axes[1, 1].set_title("Substitutions Only (Pearson $r$)")

plt.suptitle("Prime Editing Prediction Correlation Heatmaps (Track A)", y=0.98)
plt.tight_layout()
plt.show()

In [ ]:
# Multi-model regression line plots for All Edit Types
domains = {
    'PE2max': ('Normalized+3rep_HEK-M-1-7D+pe_ratio_%', 'pred_dp_base'),
    'PE2max-dRNaseH': ('Normalized+3rep_HEK-M-2-7D+pe_ratio_%', 'pred_dp6_drnaseh'),
    'PE6a': ('Normalized+3rep_HEK-M-3-7D+pe_ratio_%', 'pred_dp6_pe6a'),
    'PE6b': ('Normalized+3rep_HEK-M-4-7D+pe_ratio_%', 'pred_dp6_pe6b'),
    'PE6c': ('Normalized+3rep_HEK-M-5-7D+pe_ratio_%', 'pred_dp6_pe6c')
}

fig, axes = plt.subplots(1, 5, figsize=(25, 6), sharey=True)

for i, (domain_name, (gt_col, dp6_col)) in enumerate(domains.items()):
    ax = axes[i]
    unique_cols = list(set([gt_col, dp6_col, 'pred_dp_base', 'pred_pridict2_hek']))
    sub_df = df[unique_cols].dropna()
    
    # Background scatter using light blue
    ax.scatter(sub_df[dp6_col], sub_df[gt_col], alpha=0.12, color='#B0C4DE', s=12, label='Measured Data')
    
    # PRIDICT2.0 line (Light blue)
    sns.regplot(
        data=sub_df, x='pred_pridict2_hek', y=gt_col, ax=ax, scatter=False,
        color='#A0C4DF', line_kws={'linewidth': 2, 'label': 'PRIDICT2.0 Regression ($\rho$={:.3f})'.format(res_all.loc[i, 'Spearman_PRIDICT2'])}
    )
    
    # DeepPrime-Base line (Steel blue)
    sns.regplot(
        data=sub_df, x='pred_dp_base', y=gt_col, ax=ax, scatter=False,
        color='#4682B4', line_kws={'linewidth': 2, 'label': 'DeepPrime-Base Regression ($\rho$={:.3f})'.format(res_all.loc[i, 'Spearman_DeepPrime_Base'])}
    )
    
    # DeepPrime6 line (Dark navy, skipped for PE2max)
    if domain_name != 'PE2max':
        sns.regplot(
            data=sub_df, x=dp6_col, y=gt_col, ax=ax, scatter=False,
            color='#08306B', line_kws={'linewidth': 2.5, 'label': 'DeepPrime6 Regression ($\rho$={:.3f})'.format(res_all.loc[i, 'Spearman_DeepPrime6'])}
        )
    
    ax.set_title(f"{domain_name} Domain", fontweight='bold')
    ax.set_xlabel("Predicted Efficiency (%)")
    if i == 0:
        ax.set_ylabel("Measured Editing Efficiency (%)")
    else:
        ax.set_ylabel("")
        
    ax.legend(loc='lower right', frameon=True, facecolor='white', framealpha=0.9)

plt.suptitle("On-Target Prime Editing Efficiency vs. Model Predictions (All Edit Types)", y=1.02)
plt.tight_layout()
plt.show()